In [1]:
# ----------------------------
# Imports & configuration
# ----------------------------
import sys
import glob
import re
import h5py
import dataclasses
import sys
import glob

import os
from tqdm import tqdm
import importlib

from __future__ import annotations

from pathlib import Path

import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

In [2]:
# ----------------------------
# Imports & configuration
# ----------------------------

#Note to user: you most likely want higher resolution data with resoanble timestepping. In my experience I have found timesteps of about 1/100 of the eddyturnover time to work well.
#In this example eddy turnover time is about 1.5 and the timesteps are chosen to be .01

INP = Path("/storage/home/hcoda1/4/mugliotti3/scratch/temporary/DNS_SPIDER.nc")   #where DNS is saved
ds = xr.open_dataset(INP)                #import DNS
w0 = np.asarray(ds["w"].values)
Nt, Nx, Ny = w0.shape
print(Nx, Ny, Nt)

dx = 2*np.pi/Nx 
dy = 2*np.pi/Ny
dt = float(ds.attrs["save_every"])
print(dx, dy, dt)

Delta = 6*np.pi/256                 #Feel free to play with Delta

2048 2048 50
0.0030679615757712823 0.0030679615757712823 0.01


In [3]:
# ----------------------------
# Some functions we need
# ----------------------------

def fourier_filter_np(field: np.ndarray, Delta: float) -> np.ndarray:
    field = np.asarray(field)
    Nx, Ny = field.shape[:2]

    # integer wavenumber indices consistent with np.fft.fft2 ordering
    qx = np.fft.fftfreq(Nx) * Nx  # [0,1,...,Nx/2,-Nx/2+1,...,-1]
    qy = np.fft.fftfreq(Ny) * Ny

    KX, KY = np.meshgrid(qx, qy, indexing="ij")
    R2 = KX**2 + KY**2

    G = np.exp(-(R2) * (Delta**2) / 24.0)

    # Apply filter with vectorized FFTs over trailing dims
    if field.ndim == 2:
        F = np.fft.fft2(field, s=(Nx, Ny))
        out = np.real(np.fft.ifft2(F * G, s=(Nx, Ny)))
        return out

    elif field.ndim == 3:
        # (Nx,Ny,K)
        F = np.fft.fft2(field, s=(Nx, Ny), axes=(0, 1))
        out = np.real(np.fft.ifft2(F * G[..., None], s=(Nx, Ny), axes=(0, 1)))
        return out

    elif field.ndim == 4:
        # (Nx,Ny,K,C)
        F = np.fft.fft2(field, s=(Nx, Ny), axes=(0, 1))
        out = np.real(np.fft.ifft2(F * G[..., None, None], s=(Nx, Ny), axes=(0, 1)))
        return out

    else:
        raise ValueError("ERROR: field must have 2D, 3D, or 4D shape")

def vorticity_to_velocity_np(w0: np.ndarray, Lx: float = 2*np.pi, Ly: float = 2*np.pi):
    w0 = np.asarray(w0, dtype=np.float64)
    if w0.ndim != 3:
        raise ValueError(f"Expected w0 with shape (Nt,Nx,Ny), got {w0.shape}")
    Nt, Nx, Ny = w0.shape
    dx = Lx / Nx
    dy = Ly / Ny
    kx = 2.0 * np.pi * np.fft.fftfreq(Nx, d=dx)   # (Nx,)
    ky = 2.0 * np.pi * np.fft.fftfreq(Ny, d=dy)   # (Ny,)
    KX = kx[:, None]
    KY = ky[None, :] 
    K2 = KX**2 + KY**2
    w_hat = np.fft.fft2(w0, axes=(1, 2))  # (Nt,Nx,Ny)
    K2_safe = K2.copy()
    K2_safe[0, 0] = 1.0
    psi_hat = -w_hat / K2_safe[None, :, :]
    psi_hat[:, 0, 0] = 0.0  # set mean mode to 0
    u_hat = (1j * KY[None, :, :]) * psi_hat
    v_hat = (-1j * KX[None, :, :]) * psi_hat
    U = np.real(np.fft.ifft2(u_hat, axes=(1, 2)))  # (Nt,Nx,Ny)
    V = np.real(np.fft.ifft2(v_hat, axes=(1, 2)))

    return U, V  

def pressure_poisson_from_velocity_np(u_bar: np.ndarray, Lx: float = 2*np.pi, Ly: float = 2*np.pi):
    u_bar = np.asarray(u_bar, dtype=np.float64)
    if u_bar.ndim != 4 or u_bar.shape[-1] != 2:
        raise ValueError(f"Expected u_bar with shape (Nx,Ny,Nt,2), got {u_bar.shape}")
    Nx, Ny, Nt, _ = u_bar.shape
    dx = Lx / Nx
    dy = Ly / Ny
    kx = 2.0 * np.pi * np.fft.fftfreq(Nx, d=dx)  # (Nx,)
    ky = 2.0 * np.pi * np.fft.fftfreq(Ny, d=dy)  # (Ny,)
    KX = kx[:, None]                              # (Nx,1)
    KY = ky[None, :]                              # (1,Ny)
    K2 = KX**2 + KY**2
    u = u_bar[..., 0]  # (Nx,Ny,Nt)
    v = u_bar[..., 1]  # (Nx,Ny,Nt)
    uu = u * u
    uv = u * v
    vv = v * v
    UU_hat = np.fft.fft2(uu, axes=(0, 1))  # (Nx,Ny,Nt)
    UV_hat = np.fft.fft2(uv, axes=(0, 1))
    VV_hat = np.fft.fft2(vv, axes=(0, 1))
    rhs_hat = (KX**2)[:, :, None] * UU_hat + 2.0 * (KX * KY)[:, :, None] * UV_hat + (KY**2)[:, :, None] * VV_hat
    K2_safe = K2.copy()
    K2_safe[0, 0] = 1.0
    p_hat = -rhs_hat / K2_safe[:, :, None]
    p_hat[0, 0, :] = 0.0  # zero-mean pressure gauge
    p_bar = np.real(np.fft.ifft2(p_hat, axes=(0, 1)))  # (Nx,Ny,Nt)
    return p_bar

In [4]:
# ----------------------------
# Calculate u and p filtered as well as u' for R
# ----------------------------

Nt, Nx, Ny = w0.shape  
U_txy, V_txy = vorticity_to_velocity_np(w0, Lx=2*np.pi, Ly=2*np.pi)  

U = np.transpose(U_txy, (1, 2, 0))  
V = np.transpose(V_txy, (1, 2, 0))  

u = np.stack((U, V), axis=-1)       
u_bar = fourier_filter_np(u, Delta=Delta)  
u_dash = u - u_bar
u_dash_bar = fourier_filter_np(u_dash, Delta=Delta)  

In [5]:
# ----------------------------
# Define Reynolds stress tensor
# ----------------------------

R_xx = fourier_filter_np(u_dash[:,:,:,0]**2, Delta)-u_dash_bar[:,:,:,0]**2 
R_xy = fourier_filter_np(u_dash[:,:,:,0]*u_dash[:,:,:,1], Delta)-u_dash_bar[:,:,:,0]*u_dash_bar[:,:,:,1]
R_yy = fourier_filter_np(u_dash[:,:,:,1]**2, Delta)-u_dash_bar[:,:,:,1]**2 
R = np.stack([np.stack([R_xx, R_xy], axis=-1),  np.stack([R_xy, R_yy], axis=-1)], axis=-2)
del R_xx, R_xy, R_yy

In [6]:
# ----------------------------
# Define the library
# ----------------------------

#Note: Picking correct hyperparameters is very hard and something that requires lots of training and developing intuition.
#I would argue, it's easier to just try a few different values at first rather than trying to find the perfect value beforehand.
#Also always feel free to reach out to me if it's not working out: matteougliotti@gatech.edu

#I also recommend not including pressure as when it is fairly constant we find the linear term p*R_ab which is not the case though in datasets where its changing a lot. Also units don't match

from utils import save, load
from library import *
from continuous_process_library_terms import *
from commons_utils import *
from commons_identify_models import *

Uobs = Observable(string='u', rank=1)
Tobs = Observable(string='R', rank=2, can_commute_indices = True, antisymmetric = False)
observables = [Uobs,Tobs]
data_dict = {'u' : u_bar ,'R': R}

np.random.seed(1)
world_size = np.array(u_bar.shape[:3])
pad = 0
# fix random seed
dxs = [dx, dy, dt]
max_observable_counts = {Uobs: 2, Tobs:1}
srd = SRDataset(world_size=world_size, data_dict=data_dict, observables=observables, dxs=dxs, 
                irreps=SRDataset.all_rank2_irreps(), cache_primes=True)
srd.make_libraries(max_complexity=4, max_observable_counts= max_observable_counts, max_dt = 1)

In [ ]:
# ----------------------------
# Calculate the library values
# ----------------------------

import os

num_cores = os.cpu_count()
print(f"Logical CPU cores available: {num_cores}")
Ndomainz = len(srd.libs[SymmetricTraceFree(rank=2)].terms)*10
num_processorz = num_cores - 4
Ndomainz_rounded = num_processorz * round(Ndomainz / num_processorz)
print(len(srd.libs[SymmetricTraceFree(rank=2)].terms))
print(num_processorz)
print(Ndomainz_rounded)
dom_width = 32
dom_time = 30 
pad = 0
srd.make_domains(ndomains=Ndomainz_rounded, domain_size=[dom_width, dom_width, dom_time], pad=pad)
srd.make_weights(m=12, qmax=0)
srd.set_LT_scale(L=dx*dom_width, T=dt*dom_time) # note that this line must go before make_library_matrices
srd.make_library_matrices(debug=False, parallel=True, num_processors=num_processorz)  # or whatever number of cores you want

Logical CPU cores available: 128
31
124
248


In [ ]:
# ----------------------------
# Look at the terms you are interested in and not down their index (here e.g. 15 for dtR)
# ----------------------------

srd.libs[srd.irreps[3]].terms

In [ ]:
# ----------------------------
# Run the regression
# ----------------------------

from commons_identify_models import *
import copy

libs = srd.libs
lib3 = libs[srd.irreps[3]]

for aaaa in range(1,6):
    max_kk = 6
    print(aaaa)
    reg_opts_list = []

    # for regression we now need to construct a Scaler, Initializer, ModelIterator, and Threshold
    scaler = Scaler(sub_inds=None, char_sizes=lib3.col_weights, row_norms=None, train_fraction=1, unit_rows=True)
    #init = Initializer(method='combinatorial', start_k=2)
    #init = Initializer(method='combinatorial', start_k=9999)
    init = Initializer(method='power', start_k=max_kk)
    #res = Residual(residual_type='fixed_column', anchor_col=0)
    #res = Residual(residual_type='dominant_balance')
    res = Residual(residual_type='dominant_balance')

    iterator = ModelIterator(max_k=max_kk, backward_forward=True, max_passes=20)
    thres = Threshold(threshold_type='jump', gamma=1.5, delta=1e-6, n_terms=aaaa)
    #thres = Threshold(threshold_type='information', ic=AIC)

    opts = {'scaler': scaler, 'initializer': init, 'residual': res,
            'model_iterator': iterator, 'threshold': thres}
    opts['verbose'] = False
    opts['inhomog'] = True
    opts['inhomog_col'] = 15 #25

    reg_result = sparse_reg_bf(lib3.Q, **opts)
    zipped = [(lib3.terms[i], c) for i, c in enumerate(reg_result.xi) if c != 0]
    eqn = Equation([e[0] for e in zipped], [e[1] for e in zipped])

    print(eqn, "; residual:", reg_result.lambd)